# AgriWatch Monitor — woda w winnicy (Condom, Gers)
Notatnik zarządczy całego potoku. Otwieraj z GitHub (Plik → Otwórz → GitHub), potem `Uruchom wszystko`.
GPU potrzebne tylko przy nowych scenach (SEN2SR).

Łańcuch: Sentinel-2, ERA5-Land i satelitarny SWI (Copernicus) → SEN2SR 2,5 m z kontrolą → indeksy winnicy →
statystyki 15 winnic (W3b) → anomalie i status dekadowy (logika EDO CDI) → walidacja na ISMN Condom → pogoda
(Météo-France) → testy W3b → biuletyn → **dashboard z kartami „Moja winnica” i „Woda w glebie”** → raport.

Stan i plan: `docs/STAN_I_PLAN.md`; „Moja winnica”: `docs/plans/W3b_plan_wdrozenia.md`, protokół
`docs/evidence/W3b_protokol_walidacji.md` (zarejestrowany przed liczeniem).
Klucze: tylko z `MyDrive/GeoWorldLook/secrets/` (np. `cdse.env`); notatnik nie wypisuje ich treści.
Wyniki: `MyDrive/GeoWorldLook/agriwatch/data/05_Final_Outputs/agriwatch/` i tabele `data/registry/gwl_*.csv`.


## 1. Dysk Google i kod z GitHub


In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = 'https://github.com/geoworldlook/agriscreen.git'
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/agriwatch'
if IN_COLAB:
    if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
        !git clone {REPO_URL} "{PROJECT_DIR}"
    # Lokalne zmiany w kodzie (np. zapisany notatnik) blokowały `pull --ff-only` i kod zostawał stary:
    # odkładamy je do stash (do odzyskania: git stash list / git stash pop), potem pull.
    !git -C "{PROJECT_DIR}" stash push -q -m "colab-autostash" || true
    !git -C "{PROJECT_DIR}" pull --ff-only
    !git -C "{PROJECT_DIR}" log -1 --format="Kod: %h %ci %s"
    if os.popen(f'git -C "{PROJECT_DIR}" rev-parse HEAD').read() != os.popen(f'git -C "{PROJECT_DIR}" rev-parse origin/main').read():
        print('[UWAGA] Kod na Dysku różni się od origin/main — wyniki będą z innej wersji kodu.')
else:
    PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import importlib
sys.modules.setdefault('imp', importlib)
try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print(f'[INFO] autoreload pominięty ({e})')
print('Projekt:', PROJECT_DIR)


## 2. Środowisko: pakiety, GEE, rejestr, konfiguracja


In [ ]:
from step_05_colab_run import (setup_runtime, monitor_config, run_task, run_history, registry_summary,
                               task_ingest_s2, task_ingest_era5, task_ingest_swi, task_scene_stats, task_anomalies,
                               task_validate, task_weather, task_ref_validate, task_bulletin, task_dashboard,
                               task_summary, task_parcel_stats, task_parcel_tests, task_parcel_anomalies, task_zones,
                               task_months, task_panel)
from step_06_dashboard import show_dashboard
rt = setup_runtime(PROJECT_DIR, gee_project='ee-geoworldlook', pull=False)
cfg = monitor_config(rt)   # zmiany: monitor_config(rt, {'VINEYARDS': {'VINEYARD_06': 6, 'VINEYARD_07': 7}})


## 3. Dane: rastry Sentinel-2 (przyrostowo) i ERA5-Land 1991 → dziś


In [ ]:
run_task(rt, 'ingest_s2', task_ingest_s2, rt, cfg)
run_task(rt, 'ingest_era5', task_ingest_era5, rt, cfg)


## 3b. Satelitarny SWI — pobieranie (Plan v9 W3a)
CLMS Soil Water Index z Copernicus Data Space (klucze z `secrets/cdse.env`, bez wypisywania): **0,1° v4** (radar ASCAT,
I 2007 – dziś, jedna wersja produktu) do wiersza satelity na karcie wody i **1 km v2** (od 13 VII 2025), zbierany bez
normy, dopóki nie ma dłuższej historii. Pierwsze uruchomienie pobiera archiwum od 2007 r. (kilka minut), kolejne tylko
ostatnie dni. Metoda: `docs/evidence/W3a_SWI_1km_satelita.md` §0 (T = 15, norma 2007–2024).


In [ ]:
run_task(rt, 'ingest_swi', task_ingest_swi, rt, cfg)


## 4. Indeksy winnicy i stacji: 10 m oraz SEN2SR 2,5 m (kontrola H-SR0/H-SR1, bez zastępstwa interpolacją)


In [ ]:
run_task(rt, 'scene_stats', task_scene_stats, rt, cfg)


## 4b. Moja winnica: statystyki 15 winnic (W3b)
Średnie wskaźników (NDVI, NDMI, NDRE, CRSWIR) i pasm B04, B06, B8A, B12 dla wszystkich 15 winnic, z pasem 20 m od
brzegu odciętym (protokół W3b §1). Liczone z archiwum na Dysku, bez pobierania i bez GPU. Pierwsze uruchomienie
przelicza całe archiwum (ok. 5–15 min, zapis co 25 scen, więc przerwana sesja wznawia się sama), kolejne tylko nowe
sceny. Komunikat podaje, ile winnic spełnia próg 40 pikseli (oczekiwane 13 z 15).


In [ ]:
run_task(rt, 'parcel_stats', task_parcel_stats, rt, cfg)


## 5. Anomalie, status, walidacja, przymrozki i upały
`weather`: dane dobowe Météo-France (Gers, Lot-et-Garonne) i ERA5-Land w punktach najbliższych stacji — walidacja
Tmin, Tmax, opadu i SPI oraz kalibracja progów przymrozku i upału. Pierwsze uruchomienie pobiera ERA5 dla stacji
(kilka minut), kolejne korzystają z cache.


In [ ]:
run_task(rt, 'anomalies', task_anomalies, rt, cfg)
run_task(rt, 'validate', task_validate, rt, cfg)
run_task(rt, 'weather', task_weather, rt, cfg)


## 5b. Walidacja na stacjach referencyjnych (Plan v9 W2 + REF)
Jeden protokół, zarejestrowany przed liczeniem (`docs/evidence/W2_REF_protokol_walidacji.md`, D-083): ERA5-Land
0–100 cm i satelitarny SWI wobec pomiarów w gruncie na trzech miejscach — **Condom** (ISMN 20–30 cm po kontroli
czujników A4), **Auradé** w Gers (ICOS, 2019–2025) i **Lusignan** (ICOS, dane bieżące). Pierwsze uruchomienie pobiera
dane ICOS oraz ERA5 i SWI w punktach stacji (kilkanaście minut), kolejne korzystają z cache. Wyniki w raporcie (sekcja 7)
i w `validation_reference.csv`. **Prześlij wynik** — na jego podstawie uzupełnimy „Na ile ufać” na karcie.


In [ ]:
run_task(rt, 'ref_validate', task_ref_validate, rt, cfg)


## 5c. Moja winnica: testy zamrożone i anomalie działek (W3b) — prześlij wynik
`parcel_tests` liczy testy według zarejestrowanego protokołu (`docs/evidence/W3b_protokol_walidacji.md`) na scenach
do daty zamrożenia (pierwsze uruchomienie zapisuje ją w `w3b_freeze.json`): T1 kalibracja, T2 szum, **T3 zgodność
roślinności okolicy z czujnikiem w gruncie (kryterium zdania 3)**, wariant wielopasmowy (kryterium przyjęcia) i
przeliczone wartości „przed” z F4. Kolejne uruchomienia zwracają „skipped” — liczby są zamrożone.
`parcel_anomalies` liczy anomalie wszystkich winnic do karty bieżącej.
**Prześlij komunikat `parcel_tests` i plik `w3b_tests.csv`** — na nich powstanie notatka wyników W3b.


In [ ]:
run_task(rt, 'parcel_tests', task_parcel_tests, rt, cfg)
run_task(rt, 'parcel_anomalies', task_parcel_anomalies, rt, cfg)


## 5d. Strefa gorsza niż zwykle (W3c) — prześlij wynik
Według zarejestrowanego protokołu (`docs/evidence/W3c_protokol_strefy.md`, D-090): w każdej winnicy piksel minus
mediana działki (wygładzone do 30 m), porównane z tym samym miejscem w 5 poprzednich latach. Próg mapy wyznacza test
szumu na parach zdjęć bez deszczu. Warstwa „Zones vs usual” trafia do panelu **tylko po spełnieniu trzech warunków**
(§6). Pierwsze uruchomienie czyta całe archiwum (kilka–kilkanaście minut, zapis co 50 scen), kolejne tylko nowe sceny;
data zamrożenia testów trafia do `w3c_freeze.json`.
**Prześlij komunikat `zones` i plik `w3c_tests.csv`** — na nich powstanie notatka wyników W3c.


In [ ]:
run_task(rt, 'zones', task_zones, rt, cfg)


## 5e. Miejsce najniżej od 5 lat (W3d) — prześlij wynik
Według zarejestrowanego protokołu (`docs/evidence/W3d_protokol_miesiace.md`, D-091): dla każdego kwadratu 30 m
w winnicy wartość miesiąca (mediana z czystych dni) porównana z tym samym miesiącem w 5 poprzednich latach tego
samego miejsca; po przesadzeniu (rejestr RPG, `data/rpg_winnice.csv`) historia liczy się od nowa. Warstwa „Months”
trafia do panelu tylko, gdy test D1 pokaże, że „najniżej od 5 lat” w zwykłe miesiące zdarza się mniej więcej tak
rzadko, jak powinno (ok. 1 na 6). Pierwsze uruchomienie czyta całe archiwum (ok. 1–2 min).
**Prześlij komunikat `months` i plik `w3d_tests.csv`.**


In [ ]:
run_task(rt, 'months', task_months, rt, cfg)


## 6. Biuletyn i dashboard winnicy
Od góry: **karta „Moja winnica”** (gleba w okolicy, wskaźnik wody w roślinności Twojej działki wobec jej historii,
okolica — tylko po spełnionym kryterium testu), **karta „Woda w glebie”** (kategoria suszy, niedobór, miejsce wśród
lat, od kiedy, opad, prawdopodobieństwo, „Na ile ufać”), krzywa sezonu, pogoda, prognoza, przymrozki i upały; niżej
„Roślinność działki (informacyjnie)” (mapa NDVI, kondycja) i „Szczegóły” (status EDO, wiarygodność, matryca sygnałów,
ostatnie lata). Biuletyn (`bulletin.md`, `agriwatch_latest.json`) powstaje po testach W3b, więc zawiera kartę działki.
Plik: `data/05_Final_Outputs/agriwatch/dashboard.html` (można otworzyć w przeglądarce z Dysku).


In [ ]:
run_task(rt, 'bulletin', task_bulletin, rt, cfg)
run_task(rt, 'dashboard', task_dashboard, rt, cfg)
p = os.path.join(cfg['OUTPUT_DIR'], 'dashboard.html')
if os.path.exists(p):
    show_dashboard(p)
else:
    print('Dashboard nie powstał — sprawdź komunikat zadania powyżej.')


## 6b. Panel AgriScreen — 15 winnic (po angielsku, do publikacji na blogu)
Układ jak Wago (ESA WineEO): lista winnic, mapa na zdjęciu satelitarnym Esri z półprzezroczystą warstwą — piksele
10 m (NDVI, NDRE, NDMI), strefy wobec reszty działki, **miejsca najniżej / najwyżej od 5 lat (W3d) i strefy wobec
zwykłego stanu (W3c) — tylko po przejściu testu**
albo cała działka wobec własnej historii; statystyki, gleba, pogoda (prognoza Open-Meteo), karta, sezon i „How far to
trust it”. Plik `panel.html` (Leaflet z unpkg, reszta w pliku) trafia na blog GeoWorldLook; `panel_data.json` czyta
notatnik `notebooks/AgriScreen_story.ipynb`.


In [ ]:
run_task(rt, 'panel', task_panel, rt, cfg)
p = os.path.join(cfg['OUTPUT_DIR'], 'panel.html')
if os.path.exists(p):
    show_dashboard(p, height=3600)


## 7. Raport z uruchomienia (do analizy)
`run_summary.md` — wersje, konfiguracja, pokrycie danych, kontrola SR, **karty „Woda w glebie” i „Moja winnica”
z „Na ile ufać”, testy W3b**, status per rok, walidacja, pogoda, błędy.
**Po każdym większym uruchomieniu prześlij ten plik** (albo wklej jego treść) — na nim opieramy wnioski.


In [ ]:
run_task(rt, 'summary', task_summary, rt, cfg)
print(open(os.path.join(cfg['OUTPUT_DIR'], 'run_summary.md'), encoding='utf-8').read())


## 8. Rejestr i historia uruchomień


In [ ]:
display(registry_summary(rt))
display(run_history(rt, n=10))
